# ESMFold2: Structure Prediction and Inversion Gradients

![ESMFold2](https://proto-bio.github.io/proto-assets/images/tool/esmfold2/hero.png)

This notebook demonstrates all-atom biomolecular complex structure prediction using **ESMFold2** from Biohub, and then inverts the same model to differentiate structural objectives with respect to a designed sequence. ESMFold2 folds proteins, DNA, RNA, small-molecule ligands, and their complexes via a diffusion-based structure module conditioned on an ESM protein language model. Two checkpoints are available: `esmfold2-fast` (default), an inference-optimized single-sequence variant, and `esmfold2`, a larger MSA-capable variant for challenging targets.

- Paper: https://biohub.ai/papers/esm_protein.pdf
- Upstream: https://github.com/Biohub/esm

In [1]:
from pathlib import Path

from proto_tools.tools.structure_prediction.esmfold2 import (
    ESMFold2Config,
    ESMFold2Input,
    ESMFold2Output,
    run_esmfold2,
)
from proto_tools.entities.complex import Chain, Complex
from proto_tools.entities.ligands import Fragment
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Input schema: a list of biomolecular complexes plus optional pre-computed MSAs.
display_api_reference("esmfold2-prediction", "input", "run_esmfold2")

**Input** — `ESMFold2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [3]:
# Config schema: checkpoint selection, refinement loops, sampling steps, MSA toggle.
display_api_reference("esmfold2-prediction", "config", "run_esmfold2")

**Config** — `ESMFold2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1200</code> | Maximum execution time in seconds. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>use_msa</code> | <code>bool</code> | <code>False</code> | Generate MSAs via MMseqs2 for protein chains. Only valid with model_checkpoint='esmfold2'. |
| <code>msa_search_config</code> | <code>Mmseqs2HomologySearchConfig &#124; None</code> | <code>None</code> | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| <code>pair_heterocomplex_msas</code> | <code>bool</code> | <code>True</code> | Whether heterocomplex protein chains should use taxonomy-paired MSA generation. |
| <code>model_checkpoint</code> | <code>Literal['esmfold2', 'esmfold2-fast']</code> | <code>'esmfold2-fast'</code> | 'esmfold2' is the larger MSA-capable variant; 'esmfold2-fast' is single-sequence and faster. |
| <code>num_loops</code> | <code>int</code> | <code>20</code> | Iterative refinement loops through the model. Higher = more accurate but slower. |
| <code>num_sampling_steps</code> | <code>int</code> | <code>100</code> | Diffusion sampling steps for the structure module. Higher = more refined but slower. |
| <code>diffusion_samples</code> | <code>int</code> | <code>1</code> | Independent diffusion samples per complex; the highest-pLDDT sample is returned. |
| <code>step_scale</code> | <code>float &#124; None</code> | <code>None</code> | Diffusion step size override (typical 1.0-2.0). None uses upstream default; lower = more diversity. |
| <code>noise_scale</code> | <code>float &#124; None</code> | <code>None</code> | Diffusion noise scale override. None uses the upstream sampler default. |
| <code>max_inference_sigma</code> | <code>float &#124; None</code> | <code>None</code> | Maximum sigma value for the diffusion sampler. None uses the upstream default (256.0). |
| <code>early_exit</code> | <code>bool</code> | <code>False</code> | Exit refinement loops early when convergence is detected. |

In [4]:
# Output schema: one Structure per input complex, each carrying ESMFold2Metrics.
display_api_reference("esmfold2-prediction", "output", "run_esmfold2")

**Output** — `ESMFold2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | List of predicted structures, one per input complex. |

**Metrics** — `ESMFold2Metrics` (one set per `structures` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>plddt</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>avg_pae</code> | <code>float</code> | <code>[0, 32]</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>[0, 32]</code> |  |  |

## Basic single-protein fold

Predict the structure of human ubiquitin (PDB 1UBQ, 76 residues) using the default `esmfold2-fast` single-sequence checkpoint.

In [5]:
# Human ubiquitin sequence (PDB 1UBQ, 76 residues)
ubiquitin_sequence = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

# Wrap the bare sequence in an ESMFold2Input; strings are auto-coerced to single-chain Complexes.
inputs = ESMFold2Input(complexes=[ubiquitin_sequence])

# Default config uses the inference-optimized esmfold2-fast checkpoint.
config = ESMFold2Config(device="cuda")

result = run_esmfold2(inputs, config)

Folding structures (ESMFold2):   0%|          | 0/1 [00:00<?, ?complex/s]

In [6]:
# Inspect the predicted structure and its confidence metrics.
ubq_structure = result.structures[0]

print(f"Length:           {ubq_structure.num_residues} residues")
print(f"Mean pLDDT:       {ubq_structure.metrics.plddt:.3f}")
print(f"pTM:              {ubq_structure.metrics.ptm:.3f}")
print(f"Mean PAE (Å):     {ubq_structure.metrics.avg_pae:.3f}")
print(f"B-factor column:  {ubq_structure.b_factor_type}")

Length:           76 residues
Mean pLDDT:       0.823
pTM:              0.779
Mean PAE (Å):     3.856
B-factor column:  BFactorType.PLDDT


## Multi-chain protein complex

Fold a heterodimer by passing a `Complex` with two protein chains. Multi-chain complexes additionally emit an `iptm` interface score on top of the global `ptm`.

In [7]:
# Two short helical peptides used as a toy heterodimer.
chain_a = "GSHMKQLEDKVEELLSKNYHLENEVARLKKLV"
chain_b = "GSHMEELLSKNYHLENEVARLKKLVGER"

heterodimer = Complex(
    chains=[
        Chain(sequence=chain_a, entity_type="protein"),
        Chain(sequence=chain_b, entity_type="protein"),
    ]
)

dimer_result = run_esmfold2(
    ESMFold2Input(complexes=[heterodimer]),
    ESMFold2Config(device="cuda"),
)

dimer_structure = dimer_result.structures[0]
print(f"Chains:    {len(heterodimer.chains)}")
print(f"pLDDT:     {dimer_structure.metrics.plddt:.3f}")
print(f"pTM:       {dimer_structure.metrics.ptm:.3f}")
# iptm is only populated for multi-chain complexes.
if dimer_structure.metrics.iptm is not None:
    print(f"ipTM:      {dimer_structure.metrics.iptm:.3f}")

Folding structures (ESMFold2):   0%|          | 0/1 [00:00<?, ?complex/s]

Chains:    2
pLDDT:     0.865
pTM:       0.783
ipTM:      0.769


## Protein bound to a small-molecule ligand

Small-molecule ligands can be passed as `Fragment` entries alongside biopolymer chains. The ligand is identified either by CCD code (e.g. `HEM`, `ATP`) or by SMILES.

In [8]:
# Sperm whale myoglobin (PDB 1MBN) holo form binds a heme cofactor (CCD code HEM).
myoglobin_sequence = (
    "MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKK"
    "KGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG"
)

myoglobin_complex = Complex(
    chains=[
        Chain(sequence=myoglobin_sequence, entity_type="protein"),
        Fragment(ccd_code="HEM"),
    ]
)

holo_result = run_esmfold2(
    ESMFold2Input(complexes=[myoglobin_complex]),
    ESMFold2Config(device="cuda"),
)

holo_structure = holo_result.structures[0]
print(f"pLDDT:    {holo_structure.metrics.plddt:.3f}")
print(f"pTM:      {holo_structure.metrics.ptm:.3f}")
if holo_structure.metrics.iptm is not None:
    print(f"ipTM:     {holo_structure.metrics.iptm:.3f}")

Folding structures (ESMFold2):   0%|          | 0/1 [00:00<?, ?complex/s]

pLDDT:    0.946
pTM:      0.972
ipTM:     0.980


## Inference-time compute parameters

`num_loops` controls the number of iterative refinement passes through the model, and `num_sampling_steps` controls the diffusion sampling granularity of the structure module. Both trade runtime for accuracy. The defaults (`num_loops=20`, `num_sampling_steps=100`) are a reasonable starting point; Increase them for difficult targets; decrease them for high-throughput screening.

In [9]:
# Quick screen-mode config: fewer loops and sampling steps for faster turnaround.
fast_config = ESMFold2Config(
    device="cuda",
    num_loops=1,
    num_sampling_steps=20,
)

# Higher-accuracy config: more loops and finer diffusion sampling.
accurate_config = ESMFold2Config(
    device="cuda",
    num_loops=5,
    num_sampling_steps=100,
)

fast_result = run_esmfold2(ESMFold2Input(complexes=[ubiquitin_sequence]), fast_config)
print(f"Fast pLDDT:     {fast_result.structures[0].metrics.plddt:.3f}")

Folding structures (ESMFold2):   0%|          | 0/1 [00:00<?, ?complex/s]

Fast pLDDT:     0.822


## MSA-conditioned predictions (the `esmfold2` checkpoint)

The default `esmfold2-fast` checkpoint is single-sequence. For challenging targets, switch to the larger `esmfold2` checkpoint, which can incorporate multiple sequence alignments. Two ways to supply MSAs:

1. Set `use_msa=True` on the config: ESMFold2 will run an MMseqs2 homology search automatically for each protein chain.
2. Attach pre-computed MSAs via the `msas` field on `ESMFold2Input`, one bundle per complex, keyed by chain index.

Both paths are gated on `model_checkpoint="esmfold2"`; the config validator rejects `use_msa=True` together with the single-sequence Fast checkpoint. MSA-conditioned runs are slow (the MSA search dominates) so no live cell is shown here; see the `mmseqs2-homology-search` tool for standalone MSA generation.

## Inverting the prediction: structure gradients

The same checkpoint can be run backwards. Instead of a discrete sequence, the inversion tools
take a **relaxed** distribution over a designed chain, fold the complex once, and report each
requested structural quantity together with its derivative with respect to that chain's logits.

Every quantity is read off the predicted **distogram** — the model's distribution over pairwise
distances — and every quantity is lower-is-better. This is a single evaluation, not a design
loop: stepping the logits and rescoring candidates belong to the caller.

In [10]:
from proto_tools.tools.structure_prediction.esmfold2 import (
    ESMFold2BinderCandidate,
    ESMFold2BinderGradientConfig,
    ESMFold2BinderGradientInput,
    ESMFold2Chain,
    ESMFold2ComplexCandidate,
    ESMFold2GlobularityTerm,
    ESMFold2InversionGradientConfig,
    ESMFold2InversionGradientInput,
    run_esmfold2_inversion_binder_gradient,
    run_esmfold2_inversion_gradient,
)
from proto_tools.utils import one_hot_protein_logits

### Binder design against a fixed target

`esmfold2-inversion-binder-gradient` takes a target sequence and a relaxed binder. Alongside the
binder's own foldability it offers two interface terms, `inter_contact` and `epitope`.

In [11]:
target = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ"
binder = "MKTAYIAKQRQISFVKSHFSRQLE"

binder_result = run_esmfold2_inversion_binder_gradient(
    ESMFold2BinderGradientInput(
        inputs=ESMFold2BinderCandidate(
            target_sequence=target,
            logits=one_hot_protein_logits(binder, sharpness=2.0),
        )
    ),
    ESMFold2BinderGradientConfig(
        gradient_terms=["intra_contact", "inter_contact", "globularity"],
        seed=0,
    ),
)

for term in binder_result.results[0].terms:
    rows = len(term.gradient) if term.gradient is not None else 0
    print(f"{term.term.id:<15} value={term.term_value:+.4f}  gradient rows={rows}")

Running run_esmfold2_inversion_binder_gradient [00:00]

intra_contact   value=+4.7779  gradient rows=24
inter_contact   value=+3.5715  gradient rows=24
globularity     value=+2.9092  gradient rows=24


Each quantity carries its own gradient over the same rows, so they compose directly. Weighting
is the caller's choice:

In [12]:
candidate = binder_result.results[0]
objective = candidate.compose(weights={"inter_contact": 2.0})
print(f"weighted objective: {objective.objective_loss:+.4f}")

weighted objective: +14.8301


### Several designable chains at once

`esmfold2-inversion-gradient` is the general form: a complex is a list of chains in fold order,
each either **fixed** (a discrete `sequence`) or **designable** (relaxed `logits`). Fixed chains
are folded as context and receive no gradient. Terms name the chain they evaluate through
`chain_id`, and `chain_rows` says where each designable chain sits in the concatenated gradient.

In [13]:
chain_b = "MKTAYIAKQRQISFVKS"
chain_c = "LEERLGLIEVQAPILSR"

complex_result = run_esmfold2_inversion_gradient(
    ESMFold2InversionGradientInput(
        inputs=ESMFold2ComplexCandidate(
            chains=[
                ESMFold2Chain(chain_id="A", sequence=target),
                ESMFold2Chain(chain_id="B", logits=one_hot_protein_logits(chain_b, sharpness=2.0)),
                ESMFold2Chain(chain_id="C", logits=one_hot_protein_logits(chain_c, sharpness=2.0)),
            ]
        )
    ),
    ESMFold2InversionGradientConfig(
        gradient_terms=[ESMFold2GlobularityTerm(chain_id="B"), ESMFold2GlobularityTerm(chain_id="C")],
        seed=0,
    ),
)

print("chain rows:", complex_result.results[0].chain_rows)
for term in complex_result.results[0].terms:
    print(f"{term.term.id} on chain {term.term.chain_id}: {term.term_value:+.4f}")

Running run_esmfold2_inversion_gradient [00:00]

chain rows: {'B': [0, 17], 'C': [17, 34]}
globularity on chain B: +2.2009
globularity on chain C: +2.3197


## Export results

Predicted structures export to PDB or mmCIF for downstream analysis. The B-factor column carries per-residue pLDDT confidence.

In [14]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the ubiquitin prediction to a PDB file in the output directory.
result.export(name="ubiquitin", export_path=output_dir, file_format="pdb")
print("Wrote", sorted((output_dir / "ubiquitin").glob("structure_*.pdb")))

Wrote [PosixPath('example_output/ubiquitin/structure_0.pdb')]
